# Flask ML App — Entrenamiento del modelo de diabetes

Este notebook entrena y guarda el modelo que usará la aplicación Flask.
Dataset: Pima Indians Diabetes (mismo que ejercicios anteriores).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import pickle
import urllib.request
import os

In [2]:
url  = 'https://breathecode.herokuapp.com/asset/internal-link?id=930&path=diabetes.csv'
ruta = '../data/raw/diabetes.csv'
os.makedirs(os.path.dirname(ruta), exist_ok=True)

if not os.path.exists(ruta):
    urllib.request.urlretrieve(url, ruta)

df = pd.read_csv(ruta)

# Limpiar ceros imposibles
cols_ceros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_ceros:
    df[col] = df[col].replace(0, df[col][df[col] != 0].median())

print(f'Dataset: {df.shape}')
df.head(3)

Dataset: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,125,33.6,0.627,50,1
1,1,85,66,29,125,26.6,0.351,31,0
2,8,183,64,29,125,23.3,0.672,32,1


In [3]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

clf = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42)
clf.fit(X_train_sc, y_train)

acc = accuracy_score(y_test, clf.predict(X_test_sc))
print(f'Accuracy en test: {acc:.4f}')
print(classification_report(y_test, clf.predict(X_test_sc)))

Accuracy en test: 0.7532
              precision    recall  f1-score   support

           0       0.83      0.78      0.80        99
           1       0.64      0.71      0.67        55

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.74       154
weighted avg       0.76      0.75      0.76       154



In [5]:
# Guardar modelo y scaler juntos — la app Flask los carga
os.makedirs('../models', exist_ok=True)
with open('../models/modelo_diabetes.pkl', 'wb') as f:
    pickle.dump({'modelo': clf, 'scaler': scaler, 'features': list(X.columns)}, f)

print('Modelo guardado en src/models/modelo_diabetes.pkl')

Modelo guardado en src/models/modelo_diabetes.pkl


## Enlace a producción

https://ml-app-web-with-flask-production.up.railway.app/